In [1]:
import os
from  dotenv import load_dotenv
load_dotenv()
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\Users\USER\anaconda3\envs\rag_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "rag_project_documentLoader"

In [25]:
attention_file_path = r"doc\attention.pdf"  #- scan pdf not working
epa_file_path = r"doc\epa_sample.pdf"  #- scan pdf not working
scansmpl_file_path=r"doc\scansmpl.pdf"  #-scan pdf not working
Sunny_file_path=r"doc\Sunny Farm Invoice Sample.pdf" # - working  for 
normal_file_path=r"doc\normal_pdf_example.pdf"# will work
scanned_file_path=r"doc\scanned_pdf_example.pdf"  # blank for scan pdf 

In [30]:
# not give result when scan pdf is used
print(f"epa_file_path")
epa_loader = PyMuPDFLoader(file_path=epa_file_path,) # no data read
epa_docs = epa_loader.load()
print(epa_docs)
print("----")
print(f"scansmpl_file_path: {scansmpl_file_path}")
scansmpl_loader = PyMuPDFLoader(file_path=scansmpl_file_path,) # no data read for scan pdf, but working for normal pdf
scansmpl_docs = scansmpl_loader.load()
print(scansmpl_docs)
print("----")
print(f"Sunny_file_path: {Sunny_file_path}")
Sunny_loader = PyMuPDFLoader(file_path=Sunny_file_path,)    # working
Sunny_docs = Sunny_loader.load()
print(Sunny_docs)
print("----")
print
normal_loader = PyMuPDFLoader(file_path=normal_file_path,) # working
normal_docs = normal_loader.load()
print(normal_docs)
print("----")
print
scanned_loader = PyMuPDFLoader(file_path=scanned_file_path,) #  no data read for scan pdf
scanned_docs = scanned_loader.load()
print(scanned_docs)


epa_file_path
[Document(metadata={'producer': 'Lexmark X792', 'creator': 'HardCopy', 'creationdate': '2016-02-29T11:59:58-05:00', 'source': 'doc\\epa_sample.pdf', 'file_path': 'doc\\epa_sample.pdf', 'total_pages': 3, 'format': 'PDF 1.5', 'title': 'Scanned Document', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2016-02-29T12:04:12-05:00', 'trapped': '', 'modDate': "D:20160229120412-05'00'", 'creationDate': "D:20160229115958-05'00'", 'page': 0}, page_content='SAMPLE LETTER'), Document(metadata={'producer': 'Lexmark X792', 'creator': 'HardCopy', 'creationdate': '2016-02-29T11:59:58-05:00', 'source': 'doc\\epa_sample.pdf', 'file_path': 'doc\\epa_sample.pdf', 'total_pages': 3, 'format': 'PDF 1.5', 'title': 'Scanned Document', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2016-02-29T12:04:12-05:00', 'trapped': '', 'modDate': "D:20160229120412-05'00'", 'creationDate': "D:20160229115958-05'00'", 'page': 1}, page_content=''), Document(metadata={'producer': 'Lexmark X792',

# table pdf

In [ ]:
class PyMuPDFLoader(
    file_path: str | PurePath,
    *,
    password: str | None = None,
    mode: Literal['single', 'page'] = "page",
    pages_delimiter: str = _DEFAULT_PAGES_DELIMITER,
    extract_images: bool = False,
    images_parser: BaseImageBlobParser | None = None,
    images_inner_format: Literal['text', 'markdown-img', 'html-img'] = "text",
    extract_tables: Literal['csv', 'markdown', 'html'] | None = None,
    headers: dict | None = None,
    extract_tables_settings: dict[str, Any] | None = None,
    **kwargs: Any

In [40]:
from langchain_community.document_loaders.parsers import RapidOCRBlobParser

In [89]:
table_file_path = r"doc\sample-tables.pdf"
table_loader = PyMuPDFLoader(file_path=table_file_path,mode="page",
                             extract_tables="markdown") # no data read for scan pdf
table_docs = table_loader.load()


In [102]:
import re

def extract_markdown_tables(text):
    pattern = r"(\|.*\|\n\|[-| ]+\|\n(?:\|.*\|\n?)*)"
    return re.findall(pattern, text)

In [107]:
extract_markdown_tables(table_docs[0].page_content)

['|Column header (TH)|Column header (TH)|Column header (TH)|\n|---|---|---|\n|**Row header (TH)**|Data cell (TD)|Data cell (TD)|\n|**Row header(TH)**|Data cell (TD)|Data cell (TD)|\n',
 '|Expenditure by function £ million|Col2|2009/10|2010/11 1|\n|---|---|---|---|\n|**Policy functions**|**Financial**|22.5|30.57|\n|**Policy functions**|**Information**2|10.2|14.8|\n|**Policy functions**|**Contingency**|2.6|1.2|\n|**Remunerated functions**|**Agency services** 3|44.7|35.91|\n|**Remunerated functions**|**Payments**|22.41|19.88|\n|**Remunerated functions**|**Banking**|22.90|44.23|\n|**Remunerated functions**|**Other**|12.69|10.32|\n',
 '|Main character|Daniel Radcliffe|\n|---|---|\n|**Sidekick 1**|Rupert Grint|\n|**Sidekick 2**|Emma Watson|\n|**Lovable ogre**|Robbie Coltrane|\n|**Professor**|Maggie Smith|\n|**Headmaster**|Richard Harris|']

In [103]:
def markdown_to_csv(md_table):
    lines = md_table.strip().split("\n")

    # remove separator row (|----|)
    lines = [line for i, line in enumerate(lines) if i != 1]

    csv_lines = []
    for line in lines:
        row = line.strip("|").split("|")
        row = [cell.strip() for cell in row]
        csv_lines.append(",".join(row))

    return "\n".join(csv_lines)

In [106]:
# ==============================
# 4. SAVE TABLES AS CSV
# ==============================
output_dir = "tables_csv"
os.makedirs(output_dir, exist_ok=True)

total_tables = 0

for page_num, doc in enumerate(table_docs):
    tables = extract_markdown_tables(doc.page_content)

    for table_idx, table in enumerate(tables):
        total_tables += 1

        csv_data = markdown_to_csv(table)

        file_path = os.path.join(
            output_dir,
            f"page_{page_num+1}_table_{table_idx+1}.csv"
        )

        with open(file_path, "w", encoding="utf-8") as f:
            f.write(csv_data)

print(f"✅ Total tables saved: {total_tables}")

✅ Total tables saved: 29
